# 활성화 함수

이 노트북은 `src/nn/activations.py`가 제공하는 활성화 함수 4개를 직접 실행하고 시각화하는 실습 자료이다.
이전 노트북의 변수나 실행 결과를 사용하지 않으며, 이 노트북 단독으로 완전히 실행할 수 있다.

**실행 환경**: `conda run -n numpy_py311` (GPU 불필요)

**목표**
- `sigmoid`, `softmax`, `relu`, `identity` 4개 함수의 수식과 동작을 시각화로 확인한다.
- 각 함수의 입출력 shape와 값 범위를 직접 검증한다.
- task별 출력 활성화 선택 기준을 코드로 확인한다.

## 0. 환경 설정

In [ ]:
# sys.path setup -- excluded from jupyter book build
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"project_root={PROJECT_ROOT}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from src.nn.activations import sigmoid, softmax, relu, identity

## 1. 개요

`src/nn/activations.py`는 forward pass 전용 활성화 함수 4개를 제공한다.
모두 `np.ndarray`를 입력으로 받아 같은 shape의 `np.ndarray`를 반환한다.

활성화 함수는 레이어의 선형 변환 출력(logit)에 비선형성을 추가한다.
비선형 활성화가 없으면 여러 레이어를 쌓아도 하나의 선형 변환과 동일하므로 복잡한 패턴을 학습할 수 없다.

task별 출력 활성화 선택은 다음과 같다.

| task | 출력 활성화 | 이유 |
|---|---|---|
| `multiclass` | `softmax` | 클래스 확률 합이 1이 되도록 정규화 |
| `binary` | `sigmoid` | 이진 확률을 `[0, 1]` 범위로 변환 |
| `regression` | `identity` | 원본 logit 값을 그대로 예측값으로 사용 |

## 2. Sigmoid

이진 분류 출력 레이어에 사용한다. 임의의 실수 logit을 `(0, 1)` 범위의 확률로 변환한다.

$$
\sigma(x) = \frac{1}{1 + e^{-x}}
$$

$x = 0$이면 $\sigma(0) = 0.5$이고, $x \to +\infty$이면 1에 수렴, $x \to -\infty$이면 0에 수렴한다.

**수치 안정성**: $x$가 매우 작은 음수이면 $e^{-x}$가 overflow한다.
음수 영역에서 수식을 변형하여 회피한다.

$$
\sigma(x) = \begin{cases} \dfrac{1}{1 + e^{-x}} & x \geq 0 \\[6pt] \dfrac{e^{x}}{1 + e^{x}} & x < 0 \end{cases}
$$

In [ ]:
x = np.array([-2.0, -1.0, 0.0, 1.0, 2.0], dtype=np.float32)
out = sigmoid(x)

print(f"input:  {x}")
print(f"output: {out.round(4)}")
print(f"output range: ({out.min():.4f}, {out.max():.4f})  <- must be in (0, 1)")
print(f"sigmoid(0) = {sigmoid(np.array([0.0]))[0]:.4f}  <- must be 0.5")

In [ ]:
# 수치 안정성: 극단값에서도 overflow 없이 동작해야 한다
x_extreme = np.array([-1000.0, -500.0, 500.0, 1000.0], dtype=np.float64)
out_extreme = sigmoid(x_extreme)
print(f"extreme inputs:  {x_extreme}")
print(f"extreme outputs: {out_extreme}")
print(f"no NaN: {not np.any(np.isnan(out_extreme))}")
print(f"all in (0,1): {np.all((out_extreme > 0) & (out_extreme < 1))}")

## 3. Softmax

다중 클래스 분류 출력 레이어에 사용한다. logit 벡터를 클래스 확률 분포로 변환하며, 출력의 합이 1이 된다.

$$
\text{softmax}(z)_c = \frac{e^{z_c}}{\displaystyle\sum_{k=1}^{C} e^{z_k}}
$$

**수치 안정성**: max subtraction으로 overflow를 방지한다.

$$
\text{softmax}(z)_c = \frac{e^{z_c - \max(z)}}{\displaystyle\sum_{k=1}^{C} e^{z_k - \max(z)}}
$$

In [ ]:
# 배치(N=2, C=4) 입력
logits = np.array([[2.0, 1.0, 0.1, -1.0],
                   [0.5, 0.5,  0.5,  0.5]], dtype=np.float32)
probs = softmax(logits)

print(f"input shape:  {logits.shape}")
print(f"output shape: {probs.shape}")
print()
for i in range(len(logits)):
    print(f"sample[{i}]:")
    print(f"  logits: {logits[i]}")
    print(f"  probs:  {probs[i].round(4)}")
    print(f"  sum:    {probs[i].sum():.6f}  <- must be 1.0")

In [ ]:
# 수치 안정성: 큰 값에서도 overflow 없이 동작해야 한다
x_large = np.array([[1000.0, 999.0, 998.0]], dtype=np.float64)
out_large = softmax(x_large)
print(f"large input:  {x_large}")
print(f"output:       {out_large.round(6)}")
print(f"sum:          {out_large.sum():.6f}")
print(f"no NaN: {not np.any(np.isnan(out_large))}")

# argmax는 softmax 전후로 바뀌지 않는다
logits_check = np.array([[3.0, 1.0, 2.0]], dtype=np.float32)
print(f"\nargmax before softmax: {logits_check.argmax(axis=1)}")
print(f"argmax after  softmax: {softmax(logits_check).argmax(axis=1)}")

## 4. ReLU

hidden layer의 비선형 활성화로 사용한다. 음수 입력을 0으로 만들어 네트워크에 희소성(sparsity)을 부여한다.

$$
\text{ReLU}(x) = \max(0, x) = \begin{cases} x & x > 0 \\ 0 & x \leq 0 \end{cases}
$$

양수 구간에서 gradient가 1로 일정하므로 sigmoid/tanh의 vanishing gradient 문제를 완화한다.

In [ ]:
x = np.array([-3.0, -1.0, 0.0, 1.0, 3.0], dtype=np.float32)
out = relu(x)

print(f"input:  {x}")
print(f"output: {out}")
print(f"음수 0 변환: {np.all(out[x < 0] == 0)}")
print(f"양수 보존:   {np.allclose(out[x > 0], x[x > 0])}")

## 5. Identity

activation이 필요 없는 regression 출력 레이어에 사용한다. 입력을 변환 없이 그대로 반환한다.

$$
\text{identity}(x) = x
$$

다른 activation과 동일한 인터페이스를 제공하여 task별 분기 코드 없이 통일된 방식으로 조합할 수 있다.

In [ ]:
x = np.array([-2.0, 0.0, 1.5, 10.0], dtype=np.float32)
out = identity(x)

print(f"input:  {x}")
print(f"output: {out}")
print(f"입력과 동일: {np.array_equal(x, out)}")

## 6. 시각화

4개 활성화 함수의 곡선을 나란히 시각화하여 형태 차이를 비교한다.
각 함수의 출력 범위와 꺾이는 지점을 확인한다.

In [ ]:
x = np.linspace(-5, 5, 300).astype(np.float32)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

configs = [
    ("Sigmoid",  sigmoid(x),  "steelblue",  "(0, 1)"),
    ("ReLU",     relu(x),     "tomato",     "[0, ∞)"),
    ("Identity", identity(x), "forestgreen","(-∞, ∞)"),
]

# Softmax는 다중 입력 → 단일 클래스 확률을 1D로 표현
logits_2c = np.stack([x, np.zeros_like(x)], axis=1)  # (300, 2)
softmax_class0 = softmax(logits_2c)[:, 0]

for ax, (name, y, color, rng) in zip(axes[:3], configs):
    ax.plot(x, y, color=color, linewidth=2)
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.set_title(f"{name}\nrange: {rng}", fontsize=11)
    ax.set_xlabel("x")
    ax.grid(alpha=0.3)

axes[3].plot(x, softmax_class0, color='darkorchid', linewidth=2)
axes[3].axhline(0.5, color='gray', linewidth=0.5, linestyle='--')
axes[3].axvline(0, color='gray', linewidth=0.5, linestyle='--')
axes[3].set_title("Softmax (class 0 prob\nvs 2-class logit)", fontsize=11)
axes[3].set_xlabel("logit[0] (logit[1]=0)")
axes[3].grid(alpha=0.3)

fig.suptitle("Activation Functions", fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

## 7. task별 활성화 함수 선택

`losses.py`가 activation을 내부에서 처리하므로 호출부에서 별도로 적용하지 않는다.
아래는 각 task에서 실제로 사용하는 방식을 직접 실행하여 확인한다.

In [ ]:
from src.nn.losses import cross_entropy, binary_cross_entropy, mse

np.random.seed(42)
N = 4

# multiclass: logits (N, 10) → softmax 내부 적용
logits_mc = np.random.randn(N, 10).astype(np.float32)
targets_mc = np.eye(10, dtype=np.float32)[[0, 1, 2, 3]]
loss_mc = cross_entropy(logits_mc, targets_mc)

# binary: logits (N, 1) → sigmoid 내부 적용
logits_b = np.random.randn(N, 1).astype(np.float32)
targets_b = np.array([[1.], [0.], [1.], [0.]], dtype=np.float32)
loss_b = binary_cross_entropy(logits_b, targets_b)

# regression: preds (N, 1) → identity (변환 없음)
preds_r = np.random.rand(N, 1).astype(np.float32)
targets_r = np.array([[0.1], [0.5], [0.8], [0.3]], dtype=np.float32)
loss_r = mse(preds_r, targets_r)

print(f"multiclass  cross_entropy loss: {loss_mc:.4f}")
print(f"binary      binary_cross_entropy loss: {loss_b:.4f}")
print(f"regression  mse loss: {loss_r:.4f}")
print()
print("※ activation은 각 loss 함수 내부에서 적용됨")

## 8. 검증

In [ ]:
# sigmoid
x_test = np.array([-1.0, 0.0, 1.0], dtype=np.float32)
s = sigmoid(x_test)
assert s.shape == x_test.shape
assert np.all((s > 0) & (s < 1)), "sigmoid output must be in (0, 1)"
assert abs(float(sigmoid(np.array([0.0]))[0]) - 0.5) < 1e-6, "sigmoid(0) must be 0.5"

# softmax
x_test_2d = np.array([[1.0, 2.0, 3.0], [0.0, 0.0, 0.0]], dtype=np.float32)
sm = softmax(x_test_2d)
assert sm.shape == x_test_2d.shape
assert np.allclose(sm.sum(axis=1), 1.0, atol=1e-6), "softmax row sums must be 1"
assert np.all((sm > 0) & (sm < 1)), "softmax output must be in (0, 1)"

# relu
x_test = np.array([-2.0, -1.0, 0.0, 1.0, 2.0], dtype=np.float32)
r = relu(x_test)
assert r.shape == x_test.shape
assert np.all(r >= 0), "relu output must be non-negative"
assert np.allclose(r[x_test > 0], x_test[x_test > 0]), "relu must preserve positive values"

# identity
x_test = np.array([[-1.0, 0.0, 1.0]], dtype=np.float32)
idn = identity(x_test)
assert idn.shape == x_test.shape
assert np.array_equal(idn, x_test), "identity must return input unchanged"

print("all activation validation passed")

## 요약

이 노트북에서 확인한 내용은 다음과 같다.

| 함수 | 출력 범위 | 용도 | 수치 안정성 처리 |
|---|---|---|---|
| `sigmoid` | `(0, 1)` | binary 출력, hidden layer | 음/양수 분기 처리 |
| `softmax` | `(0, 1)`, 행합=1 | multiclass 출력 | max subtraction |
| `relu` | `[0, ∞)` | hidden layer | 없음 |
| `identity` | `(-∞, ∞)` | regression 출력 | 없음 |

**핵심 설계 원칙**
- `cross_entropy`와 `binary_cross_entropy`는 내부에서 activation을 적용하므로 호출부에서 별도로 적용하지 않는다.
- `softmax`의 `argmax` 결과는 activation 전후로 동일하므로 `accuracy` 계산 시 logit에 직접 적용해도 된다.